# Data Loading and Baseline Model Training

**1. Data Loading**

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F

# Toggle for fast debugging
DEBUG = False

# 1. Transforms (no augmentation for now)
transform_train = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]
)

transform_test = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]
)

# 2. Datasets
trainset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform_train
)

testset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform_test
)

# Debug subset
if DEBUG:
    trainset = torch.utils.data.Subset(trainset, range(10000))

# 3. DataLoaders
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True, num_workers=4, pin_memory=True
)

testloader = torch.utils.data.DataLoader(
    testset, batch_size=128, shuffle=False, num_workers=4, pin_memory=True
)

# Optional debug print to check classes
classes = trainset.dataset.classes if DEBUG else trainset.classes

Files already downloaded and verified
Files already downloaded and verified


**2. Instantiate the Lightweight ResNet**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# Small CNN for CIFAR-10 (fast + easy for sparsity)
class SmallCNN(nn.Module):
    def __init__(self):
        super(SmallCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 32x32 → 16x16
        x = self.pool(F.relu(self.conv2(x)))  # 16x16 → 8x8
        x = self.pool(F.relu(self.conv3(x)))  # 8x8 → 4x4

        x = x.view(x.size(0), -1)  # flatten

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x


# Initialize model
model = SmallCNN().to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

Using device: cpu
Total parameters: 620362


**3. Baseline Model Training**

In [ ]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Choose optimizer ("sgd" or "adam")
optimizer_type = "sgd"

if optimizer_type == "sgd":
    optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
elif optimizer_type == "adam":
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

epochs = 3

print("Starting training...")

for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        # Forward
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward
        loss.backward()
        optimizer.step()

        # Metrics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100.0 * correct / total

    # ---- Evaluation ----
    model.eval()
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)

            _, predicted = outputs.max(1)
            test_total += labels.size(0)
            test_correct += predicted.eq(labels).sum().item()

    test_acc = 100.0 * test_correct / test_total

    print(
        f"Epoch [{epoch+1}/{epochs}] | "
        f"Loss: {running_loss/len(trainloader):.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Test Acc: {test_acc:.2f}%"
    )

print("Finished training!")

Starting training...
Epoch [1/3] | Loss: 1.6245 | Train Acc: 40.83% | Test Acc: 47.37%
Epoch [2/3] | Loss: 1.2909 | Train Acc: 54.60% | Test Acc: 55.61%
Epoch [3/3] | Loss: 1.1706 | Train Acc: 59.25% | Test Acc: 60.63%
Finished training!
